# 04 - Statevector and Amplitudes

This notebook looks at the quantum state before measurement and compares it with sampled counts.

Prediction-first workflow: before each run, write the amplitudes and counts you expect.

In [ ]:
from qiskit import QuantumCircuit, transpile
from qiskit.providers.basic_provider import BasicSimulator
from qiskit.quantum_info import Statevector

In [ ]:
def run_counts(circuit: QuantumCircuit, shots: int = 1024) -> dict[str, int]:
    backend = BasicSimulator()
    compiled = transpile(circuit, backend)
    return backend.run(compiled, shots=shots).result().get_counts(compiled)

def probs(counts: dict[str, int]) -> dict[str, float]:
    total = sum(counts.values())
    return {state: value / total for state, value in sorted(counts.items())}

def basis_labels(num_qubits: int) -> list[str]:
    width = max(num_qubits, 1)
    return [format(index, f'0{width}b') for index in range(2**num_qubits)]

def measured_copy(circuit: QuantumCircuit) -> QuantumCircuit:
    measured = QuantumCircuit(circuit.num_qubits, circuit.num_qubits)
    measured.compose(circuit, inplace=True)
    measured.measure(range(circuit.num_qubits), range(circuit.num_qubits))
    return measured

def inspect(circuit: QuantumCircuit, shots: int = 1024):
    sv = Statevector.from_instruction(circuit)
    labels = basis_labels(circuit.num_qubits)
    amplitudes = {str(k): complex(v) for k, v in sv.to_dict().items()}
    exact_probs = {str(k): float(v) for k, v in sv.probabilities_dict().items()}
    print(circuit.draw(output='text'))
    print('Amplitudes:', {label: amplitudes.get(label, 0j) for label in labels})
    print('Exact probabilities:', {label: exact_probs.get(label, 0.0) for label in labels})
    counts = run_counts(measured_copy(circuit), shots=shots)
    print('Sampled counts:', counts)
    print('Sampled probabilities:', probs(counts))

## Circuit A: H on `|0>`

Prediction: equal amplitudes for `|0>` and `|1>`, with counts close to 50/50.

In [ ]:
a = QuantumCircuit(1)
a.h(0)
inspect(a)

## Circuit B: H then Z

Prediction: probabilities still look 50/50, but the `|1>` amplitude has a negative sign.

In [ ]:
b = QuantumCircuit(1)
b.h(0)
b.z(0)
inspect(b)

## Circuit C: H then Z then H

Prediction: the final H converts the phase difference into a measurable `1` result.

In [ ]:
c = QuantumCircuit(1)
c.h(0)
c.z(0)
c.h(0)
inspect(c)

## Circuit D: Bell State

Prediction: only `|00>` and `|11>` have nonzero amplitudes.

In [ ]:
d = QuantumCircuit(2)
d.h(0)
d.cx(0, 1)
inspect(d)

## Reflection

1. What does the statevector show that counts do not show?
2. Why can H and H-Z produce similar counts but different states?
3. How does H-Z-H connect phase to a visible measurement result?
4. Why does this matter for search-style algorithms?